# Advanced Project Solution: Lazy `Polygon` and Lazy `Polygons`

This notebook contains an advanced, best-practice implementation for the project.

## Requirements satisfied

1. `Polygon` calculated properties are lazy and cached after first access.
2. `Polygons` no longer stores a pre-built list of polygons.
3. `Polygons` is an iterable.
4. `PolygonsIterator` is a separate iterator.
5. Iterator elements are produced lazily.

## Added value

This version also includes:

- input validation
- frozen / immutable-style objects
- `__slots__` for memory efficiency
- hashable `Polygon` objects
- total ordering for polygons
- cached `max_efficiency_polygon`
- lazy indexing and slicing for `Polygons`
- reverse iteration
- membership checks
- polygon serialization with `as_dict()` and `from_dict()`
- polygon scaling
- vertex coordinate generation
- lightweight summary generation
- stronger tests


In [1]:
import math
import numbers
import operator
from functools import total_ordering
from itertools import islice


## Reusable lazy property descriptor

The descriptor below computes a property once, stores it in an instance-level cache, and returns the cached value on later access.

This avoids repeated calculation while keeping the public API clean.


In [2]:
class LazyProperty:
    """Descriptor for cached lazy instance properties.

    The owner instance must have a `_lazy_cache` dictionary.
    """

    __slots__ = ("func", "name")

    def __init__(self, func):
        self.func = func
        self.name = func.__name__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        cache = instance._lazy_cache

        if self.name not in cache:
            cache[self.name] = self.func(instance)

        return cache[self.name]


## Helper validation functions

These helpers keep validation consistent between `Polygon` and `Polygons`.


In [3]:
def validate_integer_at_least(value, minimum, name):
    """Validate an integer value with a lower bound."""

    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(f"{name} must be an integer.")

    if value < minimum:
        raise ValueError(f"{name} must be at least {minimum}.")


def validate_finite_real(value, name):
    """Validate a finite real number."""

    if isinstance(value, bool) or not isinstance(value, numbers.Real):
        raise TypeError(f"{name} must be a real number.")

    if not math.isfinite(value):
        raise ValueError(f"{name} must be finite.")


def validate_positive_finite_real(value, name):
    """Validate a positive finite real number."""

    validate_finite_real(value, name)

    if value <= 0:
        raise ValueError(f"{name} must be positive.")


## `Polygon`

A regular polygon is fully determined by:

- `n`: number of vertices / edges
- `R`: circumradius

The object is frozen after initialization. Its derived values are computed lazily.


In [4]:
@total_ordering
class Polygon:
    """Regular polygon defined by number of vertices and circumradius.

    Parameters
    ----------
    n:
        Number of vertices / edges. Must be an integer greater than or equal to 3.
    R:
        Circumradius. Must be a positive finite real number.
    """

    __slots__ = ("_n", "_R", "_lazy_cache", "_initialized")

    def __init__(self, n, R):
        validate_integer_at_least(n, 3, "n")
        validate_positive_finite_real(R, "R")

        object.__setattr__(self, "_n", n)
        object.__setattr__(self, "_R", R)
        object.__setattr__(self, "_lazy_cache", {})
        object.__setattr__(self, "_initialized", True)

    def __setattr__(self, name, value):
        if getattr(self, "_initialized", False):
            raise AttributeError(f"{self.__class__.__name__} instances are immutable.")

        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError(f"{self.__class__.__name__} instances are immutable.")

    def __repr__(self):
        return f"Polygon(n={self._n}, R={self._R})"

    def __hash__(self):
        return hash((self.count_edges, self.circumradius))

    @LazyProperty
    def count_vertices(self):
        """Number of vertices."""
        return self._n

    @LazyProperty
    def count_edges(self):
        """Number of edges."""
        return self._n

    @LazyProperty
    def circumradius(self):
        """Circumradius of the polygon."""
        return self._R

    @LazyProperty
    def interior_angle(self):
        """Interior angle in degrees."""
        return (self._n - 2) * 180 / self._n

    @LazyProperty
    def exterior_angle(self):
        """Exterior angle in degrees."""
        return 360 / self._n

    @LazyProperty
    def central_angle(self):
        """Central angle in degrees."""
        return 360 / self._n

    @LazyProperty
    def side_length(self):
        """Length of one side."""
        return 2 * self._R * math.sin(math.pi / self._n)

    @LazyProperty
    def apothem(self):
        """Apothem / inradius of the polygon."""
        return self._R * math.cos(math.pi / self._n)

    @LazyProperty
    def inradius(self):
        """Alias for apothem."""
        return self.apothem

    @LazyProperty
    def area(self):
        """Area of the polygon."""
        return self._n / 2 * self.side_length * self.apothem

    @LazyProperty
    def perimeter(self):
        """Perimeter of the polygon."""
        return self._n * self.side_length

    @LazyProperty
    def efficiency(self):
        """Area-to-perimeter ratio."""
        return self.area / self.perimeter

    def cache_info(self):
        """Return currently cached lazy properties.

        This is useful for demonstrations and tests.
        """

        cached = tuple(sorted(self._lazy_cache))
        return {
            "cached_properties": cached,
            "count": len(cached),
        }

    def as_dict(self):
        """Serialize the polygon definition to a dictionary.

        Only the defining values are serialized, not cached derived values.
        """

        return {
            "n": self.count_vertices,
            "R": self.circumradius,
        }

    @classmethod
    def from_dict(cls, data):
        """Create a polygon from a dictionary with keys `n` and `R`."""

        try:
            n = data["n"]
            R = data["R"]
        except KeyError as ex:
            raise KeyError("data must contain keys 'n' and 'R'.") from ex

        return cls(n, R)

    def scale(self, factor):
        """Return a new similar polygon with scaled circumradius."""

        validate_positive_finite_real(factor, "factor")
        return self.__class__(self.count_vertices, self.circumradius * factor)

    def is_similar_to(self, other):
        """Return whether another polygon has the same number of vertices."""

        if not isinstance(other, Polygon):
            return False

        return self.count_vertices == other.count_vertices

    def vertices(self, *, center=(0, 0), rotation=math.pi / 2):
        """Return vertex coordinates.

        Parameters
        ----------
        center:
            `(x, y)` center of the polygon.
        rotation:
            Initial angle in radians. The default puts one vertex at the top.

        Returns
        -------
        tuple[tuple[float, float], ...]
            The polygon vertices in counterclockwise order.
        """

        try:
            center_x, center_y = center
        except Exception as ex:
            raise TypeError("center must be a two-item iterable: (x, y).") from ex

        validate_finite_real(center_x, "center_x")
        validate_finite_real(center_y, "center_y")
        validate_finite_real(rotation, "rotation")

        return tuple(
            (
                center_x + self._R * math.cos(rotation + 2 * math.pi * index / self._n),
                center_y + self._R * math.sin(rotation + 2 * math.pi * index / self._n),
            )
            for index in range(self._n)
        )

    def __eq__(self, other):
        if not isinstance(other, Polygon):
            return NotImplemented

        return (
            self.count_edges == other.count_edges
            and self.circumradius == other.circumradius
        )

    def __lt__(self, other):
        if not isinstance(other, Polygon):
            return NotImplemented

        return (
            self.count_vertices,
            self.circumradius,
        ) < (
            other.count_vertices,
            other.circumradius,
        )


## `PolygonsIterator`

The iterator owns the iteration state.

Each call to `next()` creates one `Polygon` object and then advances the iterator.


In [5]:
class PolygonsIterator:
    """Iterator for regular polygons from `start_n` through `max_n`."""

    __slots__ = ("_current_n", "_max_n", "_R")

    def __init__(self, max_n, R, start_n=3):
        validate_integer_at_least(start_n, 3, "start_n")
        validate_integer_at_least(max_n, 3, "max_n")
        validate_positive_finite_real(R, "R")

        self._current_n = start_n
        self._max_n = max_n
        self._R = R

    def __iter__(self):
        return self

    def __next__(self):
        if self._current_n > self._max_n:
            raise StopIteration

        polygon = Polygon(self._current_n, self._R)
        self._current_n += 1

        return polygon


## `Polygons`

`Polygons` is an iterable collection of regular polygons from `3` through `m` vertices.

It intentionally does **not** create or store a list of polygons.

Additional convenience methods such as indexing, slicing, summaries, and reverse iteration are still implemented lazily.


In [6]:
class Polygons:
    """Lazy iterable of regular polygons from 3 vertices through `m` vertices.

    Parameters
    ----------
    m:
        Maximum number of vertices. Must be an integer greater than or equal to 3.
    R:
        Circumradius used for every polygon. Must be a positive finite real number.
    """

    __slots__ = ("_m", "_R", "_lazy_cache", "_initialized")

    def __init__(self, m, R):
        validate_integer_at_least(m, 3, "m")
        validate_positive_finite_real(R, "R")

        object.__setattr__(self, "_m", m)
        object.__setattr__(self, "_R", R)
        object.__setattr__(self, "_lazy_cache", {})
        object.__setattr__(self, "_initialized", True)

    def __setattr__(self, name, value):
        if getattr(self, "_initialized", False):
            raise AttributeError(f"{self.__class__.__name__} instances are immutable.")

        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError(f"{self.__class__.__name__} instances are immutable.")

    def __repr__(self):
        return f"Polygons(m={self._m}, R={self._R})"

    def __len__(self):
        return self._m - 2

    def __iter__(self):
        return PolygonsIterator(self._m, self._R)

    def __reversed__(self):
        return (Polygon(n, self._R) for n in range(self._m, 2, -1))

    def __contains__(self, item):
        if not isinstance(item, Polygon):
            return False

        return (
            item.circumradius == self._R
            and 3 <= item.count_vertices <= self._m
        )

    @LazyProperty
    def max_n(self):
        """Maximum number of vertices represented by this iterable."""
        return self._m

    @LazyProperty
    def circumradius(self):
        """Circumradius used for all polygons in this iterable."""
        return self._R

    @LazyProperty
    def max_efficiency_polygon(self):
        """Polygon with the maximum area-to-perimeter ratio.

        This is computed lazily and cached.
        """

        return max(self, key=lambda polygon: polygon.efficiency)

    @LazyProperty
    def min_efficiency_polygon(self):
        """Polygon with the minimum area-to-perimeter ratio.

        This is computed lazily and cached.
        """

        return min(self, key=lambda polygon: polygon.efficiency)

    def cache_info(self):
        """Return currently cached lazy properties."""

        cached = tuple(sorted(self._lazy_cache))
        return {
            "cached_properties": cached,
            "count": len(cached),
        }

    def by_vertices(self, n):
        """Return the polygon with exactly `n` vertices.

        This creates only that one polygon.
        """

        validate_integer_at_least(n, 3, "n")

        if n > self._m:
            raise IndexError(f"n must be between 3 and {self._m}.")

        return Polygon(n, self._R)

    def __getitem__(self, index):
        """Lazy indexing and slicing.

        Integer indexing returns one polygon.

        Slicing returns a generator, not a list, so sliced values are still created lazily.
        """

        length = len(self)

        if isinstance(index, slice):
            start, stop, step = index.indices(length)
            return (Polygon(offset + 3, self._R) for offset in range(start, stop, step))

        if isinstance(index, bool):
            raise TypeError("Boolean indexes are not supported.")

        try:
            normalized_index = operator.index(index)
        except TypeError as ex:
            raise TypeError("Index must be an integer or slice.") from ex

        if normalized_index < 0:
            normalized_index += length

        if normalized_index < 0 or normalized_index >= length:
            raise IndexError("Polygons index out of range.")

        return Polygon(normalized_index + 3, self._R)

    def take(self, count):
        """Return a lazy iterator over the first `count` polygons."""

        validate_integer_at_least(count, 0, "count")
        return islice(self, count)

    def efficiency_profile(self):
        """Yield `(number_of_vertices, efficiency)` pairs lazily."""

        return (
            (polygon.count_vertices, polygon.efficiency)
            for polygon in self
        )

    def summary(self, *, ndigits=4):
        """Return a tuple of summary dictionaries.

        This materializes a summary because the caller explicitly requested it,
        but the polygons used to build it are still generated lazily.
        """

        validate_integer_at_least(ndigits, 0, "ndigits")

        return tuple(
            {
                "n": polygon.count_vertices,
                "R": polygon.circumradius,
                "side_length": round(polygon.side_length, ndigits),
                "apothem": round(polygon.apothem, ndigits),
                "area": round(polygon.area, ndigits),
                "perimeter": round(polygon.perimeter, ndigits),
                "efficiency": round(polygon.efficiency, ndigits),
            }
            for polygon in self
        )

    def closest_to_area(self, target_area):
        """Return the polygon whose area is closest to `target_area`."""

        validate_positive_finite_real(target_area, "target_area")

        return min(
            self,
            key=lambda polygon: abs(polygon.area - target_area),
        )


## Tests for `Polygon`

These tests include the original project checks and extra checks for validation, immutability, lazy caching, hashing, serialization, scaling, and vertices.


In [7]:
def test_polygon():
    abs_tol = 0.001
    rel_tol = 0.001

    invalid_inputs = [
        (2, 10, ValueError),
        (3.5, 10, TypeError),
        (True, 10, TypeError),
        (3, 0, ValueError),
        (3, -1, ValueError),
        (3, float("inf"), ValueError),
        (3, "10", TypeError),
    ]

    for n, R, expected_exception in invalid_inputs:
        try:
            Polygon(n, R)
            assert False, f"Creating Polygon({n!r}, {R!r}) should have failed."
        except expected_exception:
            pass

    n = 3
    R = 1
    p = Polygon(n, R)

    assert str(p) == "Polygon(n=3, R=1)", f"actual: {str(p)}"
    assert p.count_vertices == n, f"actual: {p.count_vertices}, expected: {n}"
    assert p.count_edges == n, f"actual: {p.count_edges}, expected: {n}"
    assert p.circumradius == R, f"actual: {p.circumradius}, expected: {R}"
    assert p.interior_angle == 60, f"actual: {p.interior_angle}, expected: 60"
    assert p.exterior_angle == 120
    assert p.central_angle == 120

    n = 4
    R = 1
    p = Polygon(n, R)

    assert p.interior_angle == 90, f"actual: {p.interior_angle}, expected: 90"

    assert math.isclose(
        p.area,
        2,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.area}, expected: 2.0"

    assert math.isclose(
        p.side_length,
        math.sqrt(2),
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.side_length}, expected: {math.sqrt(2)}"

    assert math.isclose(
        p.perimeter,
        4 * math.sqrt(2),
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.perimeter}, expected: {4 * math.sqrt(2)}"

    assert math.isclose(
        p.apothem,
        0.707,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.apothem}, expected: 0.707"

    p = Polygon(6, 2)

    assert math.isclose(p.side_length, 2, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 1.73205, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.inradius, p.apothem, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 10.3923, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 12, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 120, rel_tol=rel_tol, abs_tol=abs_tol)

    p = Polygon(12, 3)

    assert math.isclose(p.side_length, 1.55291, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 2.89778, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 27, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 18.635, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 150, rel_tol=rel_tol, abs_tol=abs_tol)

    p1 = Polygon(3, 10)
    p2 = Polygon(10, 10)
    p3 = Polygon(15, 10)
    p4 = Polygon(15, 100)
    p5 = Polygon(15, 100)

    assert p2 > p1
    assert p2 < p3
    assert p3 != p4
    assert p1 != p4
    assert p4 == p5

    # Hashing
    assert len({Polygon(4, 1), Polygon(4, 1), Polygon(5, 1)}) == 2

    # Immutability checks
    p = Polygon(5, 10)

    try:
        p._n = 99
        assert False, "Expected AttributeError when modifying _n."
    except AttributeError:
        pass

    try:
        p.new_attribute = "not allowed"
        assert False, "Expected AttributeError when adding a new attribute."
    except AttributeError:
        pass

    try:
        del p._R
        assert False, "Expected AttributeError when deleting _R."
    except AttributeError:
        pass

    # Lazy caching checks
    p = Polygon(6, 2)
    assert p.cache_info() == {"cached_properties": (), "count": 0}

    first_side_length = p.side_length
    assert "side_length" in p._lazy_cache

    second_side_length = p.side_length
    assert first_side_length == second_side_length
    assert len([key for key in p._lazy_cache if key == "side_length"]) == 1

    p = Polygon(6, 2)
    _ = p.area

    assert "area" in p._lazy_cache
    assert "side_length" in p._lazy_cache
    assert "apothem" in p._lazy_cache

    # Serialization
    p = Polygon(8, 3)
    assert p.as_dict() == {"n": 8, "R": 3}
    assert Polygon.from_dict(p.as_dict()) == p

    # Scaling
    scaled = p.scale(2)
    assert scaled == Polygon(8, 6)
    assert scaled.is_similar_to(p)

    # Vertices
    square = Polygon(4, 1)
    vertices = square.vertices(rotation=0)
    expected_vertices = (
        (1, 0),
        (0, 1),
        (-1, 0),
        (0, -1),
    )

    for actual, expected in zip(vertices, expected_vertices):
        assert math.isclose(actual[0], expected[0], rel_tol=rel_tol, abs_tol=abs_tol)
        assert math.isclose(actual[1], expected[1], rel_tol=rel_tol, abs_tol=abs_tol)

    print("Polygon tests passed.")


In [8]:
test_polygon()


Polygon tests passed.


## Tests for `Polygons`

These tests verify lazy iteration, independent iterators, no list-backed storage, slicing, reverse iteration, membership, summary generation, and cached efficiency search.


In [9]:
def test_polygons():
    invalid_inputs = [
        (2, 10, ValueError),
        (3.5, 10, TypeError),
        (True, 10, TypeError),
        (3, 0, ValueError),
        (3, -1, ValueError),
        (3, float("inf"), ValueError),
        (3, "10", TypeError),
    ]

    for m, R, expected_exception in invalid_inputs:
        try:
            Polygons(m, R)
            assert False, f"Creating Polygons({m!r}, {R!r}) should have failed."
        except expected_exception:
            pass

    polygons = Polygons(6, 10)

    assert repr(polygons) == "Polygons(m=6, R=10)"
    assert len(polygons) == 4

    # No pre-built list should exist.
    assert not hasattr(polygons, "_polygons")

    # Polygons is an iterable, not its own iterator.
    iterator = iter(polygons)

    assert iterator is not polygons
    assert iter(iterator) is iterator

    first = next(iterator)
    second = next(iterator)

    assert first == Polygon(3, 10)
    assert second == Polygon(4, 10)

    remaining = list(iterator)

    assert remaining == [
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    try:
        next(iterator)
        assert False, "Expected StopIteration."
    except StopIteration:
        pass

    # Each call to iter(polygons) returns an independent iterator.
    iterator_1 = iter(polygons)
    iterator_2 = iter(polygons)

    assert next(iterator_1) == Polygon(3, 10)
    assert next(iterator_1) == Polygon(4, 10)
    assert next(iterator_2) == Polygon(3, 10)

    # The iterable can be consumed repeatedly.
    assert list(polygons) == [
        Polygon(3, 10),
        Polygon(4, 10),
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    assert list(polygons) == [
        Polygon(3, 10),
        Polygon(4, 10),
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    # Lazy iteration: this only asks for the first two Polygon objects.
    first_two = list(islice(Polygons(100_000, 1), 2))

    assert first_two == [
        Polygon(3, 1),
        Polygon(4, 1),
    ]

    # Lazy indexing
    assert polygons[0] == Polygon(3, 10)
    assert polygons[1] == Polygon(4, 10)
    assert polygons[-1] == Polygon(6, 10)

    try:
        polygons[99]
        assert False, "Expected IndexError."
    except IndexError:
        pass

    try:
        polygons[True]
        assert False, "Expected TypeError."
    except TypeError:
        pass

    # Lazy slicing returns a generator.
    sliced = polygons[1:4]
    assert iter(sliced) is sliced
    assert list(sliced) == [
        Polygon(4, 10),
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    assert list(polygons[::-1]) == [
        Polygon(6, 10),
        Polygon(5, 10),
        Polygon(4, 10),
        Polygon(3, 10),
    ]

    assert list(reversed(polygons)) == [
        Polygon(6, 10),
        Polygon(5, 10),
        Polygon(4, 10),
        Polygon(3, 10),
    ]

    # Membership
    assert Polygon(3, 10) in polygons
    assert Polygon(6, 10) in polygons
    assert Polygon(7, 10) not in polygons
    assert Polygon(3, 99) not in polygons
    assert "not a polygon" not in polygons

    # Direct lookup by vertex count
    assert polygons.by_vertices(5) == Polygon(5, 10)

    try:
        polygons.by_vertices(7)
        assert False, "Expected IndexError."
    except IndexError:
        pass

    # take
    assert list(polygons.take(2)) == [
        Polygon(3, 10),
        Polygon(4, 10),
    ]

    # Efficiency profile
    profile = list(polygons.efficiency_profile())
    assert [item[0] for item in profile] == [3, 4, 5, 6]
    assert all(isinstance(item[1], float) for item in profile)

    # Summary
    summary = polygons.summary(ndigits=3)
    assert len(summary) == len(polygons)
    assert summary[0]["n"] == 3
    assert summary[-1]["n"] == 6
    assert "area" in summary[0]
    assert "efficiency" in summary[0]

    # max_efficiency_polygon is lazy and cached.
    polygons = Polygons(25, 10)

    assert polygons.cache_info() == {"cached_properties": (), "count": 0}

    best = polygons.max_efficiency_polygon

    assert best == Polygon(25, 10)
    assert "max_efficiency_polygon" in polygons._lazy_cache
    assert polygons.max_efficiency_polygon is best

    assert polygons.min_efficiency_polygon == Polygon(3, 10)

    # Closest area
    closest = Polygons(12, 3).closest_to_area(27)
    assert closest == Polygon(12, 3)

    # Immutability checks
    try:
        polygons._m = 99
        assert False, "Expected AttributeError when modifying _m."
    except AttributeError:
        pass

    try:
        polygons.new_attribute = "not allowed"
        assert False, "Expected AttributeError when adding a new attribute."
    except AttributeError:
        pass

    print("Polygons tests passed.")


In [10]:
test_polygons()


Polygons tests passed.


## Small usage demo

The demo below shows the collection in action.


In [11]:
polygons = Polygons(8, 5)

print(polygons)
print(f"Length: {len(polygons)}")
print(f"Initial Polygons cache: {polygons.cache_info()}")

print("\nGenerated lazily:")
for polygon in polygons:
    print(
        polygon,
        "area=",
        round(polygon.area, 4),
        "perimeter=",
        round(polygon.perimeter, 4),
        "efficiency=",
        round(polygon.efficiency, 4),
    )

print("\nBest efficiency:", polygons.max_efficiency_polygon)
print("Polygons cache after max_efficiency_polygon:", polygons.cache_info())

p = Polygon(5, 2)
print("\nPolygon before accessing calculated properties:", p.cache_info())
print("Area:", round(p.area, 4))
print("Polygon after accessing area:", p.cache_info())

print("\nFirst 3 polygons from a huge iterable:")
print(list(Polygons(1_000_000, 1).take(3)))


Polygons(m=8, R=5)
Length: 6
Initial Polygons cache: {'cached_properties': (), 'count': 0}

Generated lazily:
Polygon(n=3, R=5) area= 32.476 perimeter= 25.9808 efficiency= 1.25
Polygon(n=4, R=5) area= 50.0 perimeter= 28.2843 efficiency= 1.7678
Polygon(n=5, R=5) area= 59.441 perimeter= 29.3893 efficiency= 2.0225
Polygon(n=6, R=5) area= 64.9519 perimeter= 30.0 efficiency= 2.1651
Polygon(n=7, R=5) area= 68.4103 perimeter= 30.3719 efficiency= 2.2524
Polygon(n=8, R=5) area= 70.7107 perimeter= 30.6147 efficiency= 2.3097

Best efficiency: Polygon(n=8, R=5)
Polygons cache after max_efficiency_polygon: {'cached_properties': ('max_efficiency_polygon',), 'count': 1}

Polygon before accessing calculated properties: {'cached_properties': (), 'count': 0}
Area: 9.5106
Polygon after accessing area: {'cached_properties': ('apothem', 'area', 'side_length'), 'count': 3}

First 3 polygons from a huge iterable:
[Polygon(n=3, R=1), Polygon(n=4, R=1), Polygon(n=5, R=1)]


## Optional: summary table

The `summary()` method gives a compact, presentation-friendly view.


In [12]:
for row in Polygons(8, 5).summary(ndigits=3):
    print(row)


{'n': 3, 'R': 5, 'side_length': 8.66, 'apothem': 2.5, 'area': 32.476, 'perimeter': 25.981, 'efficiency': 1.25}
{'n': 4, 'R': 5, 'side_length': 7.071, 'apothem': 3.536, 'area': 50.0, 'perimeter': 28.284, 'efficiency': 1.768}
{'n': 5, 'R': 5, 'side_length': 5.878, 'apothem': 4.045, 'area': 59.441, 'perimeter': 29.389, 'efficiency': 2.023}
{'n': 6, 'R': 5, 'side_length': 5.0, 'apothem': 4.33, 'area': 64.952, 'perimeter': 30.0, 'efficiency': 2.165}
{'n': 7, 'R': 5, 'side_length': 4.339, 'apothem': 4.505, 'area': 68.41, 'perimeter': 30.372, 'efficiency': 2.252}
{'n': 8, 'R': 5, 'side_length': 3.827, 'apothem': 4.619, 'area': 70.711, 'perimeter': 30.615, 'efficiency': 2.31}


## Key takeaways

- `Polygon` values such as `area`, `perimeter`, `side_length`, and `apothem` are computed on first access only.
- `Polygons` does not store a list of `Polygon` instances.
- Iteration over `Polygons` creates each `Polygon` only when requested.
- Repeated iterations are safe because `Polygons.__iter__` returns a fresh iterator each time.
- Indexing and slicing are supported without list-backed storage.
- `max_efficiency_polygon` is cached after first access.
- The solution includes practical production-style conveniences while preserving the project requirements.
